# 1. 실전 예상문제
제공된 데이터(elec_train.csv)는 타이타닉호 탑승객의 정보를 담고 있다. 이를 바탕으로 생존 여부(Survived)를 예측하는 분류 모델을 개발하고, 가장 우수한 모델을 평가 데이터(titanic_test.csv)에 적용하여 생존 예측 결과를 도출하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 ROC-AUC 평가지표에 따라 평가함

[제공 데이터]
1. titanic_train.csv : 학습용 데이터 (891건)
2. titanic_test.csv : 평가용 데이터 (418건, 생존 여부 컬럼 미제공)

[데이터 컬럼]
1. PassengerId : 탑승객 고유 번호
2. Pclass : 선실 등급 (1, 2, 3)
3. Sex : 성별
4. Age : 나이
5. SibSp : 형제자매/배우자 수
6. parch : 부모/자녀 수
7. Fare : 운임
8. Embarked : 탑승항구 (C, Q, S)
9. Survived : 생존 여부 (0: 사망, 1: 생존)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred (0, 1)
3. 예측 결과 개수 : 418




In [47]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../data/titanic_train.csv')
test = pd.read_csv('../data/titanic_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.

# print(train.info())
# print(test.info())

X_train = train.drop(['PassengerId','Survived'], axis=1)
y = train['Survived']

# 반드시 test 쪽 결측치도 확인

# print(X_train.isnull().sum())
# print(test.isnull().sum())

X_train['Age'] = X_train['Age'].fillna(X_train['Age'].mean())
X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])
test['Age'] = test['Age'].fillna(test['Age'].mean())
test['Fare'] = test['Fare'].fillna(test['Fare'].mean())

# print(X_train.isnull().sum())
# print(test.isnull().sum())

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_columns = X_train.select_dtypes(exclude='object').columns
num_columns = num_columns.drop('Pclass') # 등급은 수치형이 아니니까 빼고??
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
test[num_columns] = scaler.transform(test[num_columns])

from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
# label_columns = X_train.select_dtypes(include='object').columns
# 인코더는 스케일러와 달리 1차원 전용, 위처럼 여러 컬럼 한 번에 처리 불가
X_train['Sex'] = encoder.fit_transform(X_train['Sex'])
test['Sex'] = encoder.transform(test['Sex'])
X_train['Embarked'] = encoder.fit_transform(X_train['Embarked'])
test['Embarked'] = encoder.transform(test['Embarked'])

# staritify=y로 클래스 불균형 해소
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)

# print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:,1]
# auc만 뒤에 proba
auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)
print(auc, acc, f1)

# test 데이터프레임에서도 아이디 같이 드랍해주기
test = test.drop('PassengerId', axis=1)
y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
print(result)
result.to_csv('result_13.csv', index=False)


0.8759552042160736 0.8212290502793296 0.7575757575757576
     pred
0       0
1       0
2       0
3       1
4       0
..    ...
413     0
414     1
415     0
416     0
417     1

[418 rows x 1 columns]


# 1.풀이(#1)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/titanic_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/titanic_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())
#print(train.head())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Pclass'].value_counts())
#print(train['Sex'].value_counts())
#print(train['Embarked'].value_counts())

# X, Y 데이터 셋 분리
X_train = train.drop(['PassengerId', 'Survived'], axis=1)
y = train['Survived']
X_test = test.drop(['PassengerId'], axis=1)

#print(X_train.shape, y.shape, X_test.shape)

# 결측치 처리
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].mean())
X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])

X_test['Age'] = X_test['Age'].fillna(X_test['Age'].mean())
X_test['Fare'] = X_test['Fare'].fillna(0)
#print(X_train.isnull().sum())
#print(X_test.isnull().sum())

# 수치형 변수 스케일링 - MinMaxScaler
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
num_columns = X_train.select_dtypes(exclude='object').columns
X_train[num_columns] = scaler.fit_transform(X_train[num_columns])
X_test[num_columns] = scaler.transform(X_test[num_columns])
#print(X_train.head())

# 범주형 변수 인코딩 - LabelEncoding
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
X_train['Sex'] = encoder.fit_transform(X_train['Sex'])
X_test['Sex'] = encoder.transform(X_test['Sex'])
X_train['Embarked'] = encoder.fit_transform(X_train['Embarked'])
X_test['Embarked'] = encoder.transform(X_test['Embarked'])
#print(X_train['Sex'], X_train['Embarked'])

# 학습, 검증 데이터 셋 분할
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(roc_auc, acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

0.8179183135704876 0.7932960893854749


# 1.풀이(#2)

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/titanic_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/titanic_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())
#print(train.head())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Pclass'].value_counts())
#print(train['Sex'].value_counts())
#print(train['Embarked'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['Survived'], axis=1)
y = train['Survived']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['PassengerId'], axis=1)
#print(X_full.shape)

# 결측치 처리
X_full['Age'] = X_full['Age'].fillna(X_full['Age'].mean())
X_full['Embarked'] = X_full['Embarked'].fillna(X_full['Embarked'].mode()[0])
X_full['Fare'] = X_full['Fare'].fillna(0)

#print(X_full.isnull().sum())

# 수치형 변수 스케일링 - MinMaxScaling
# Skip!!

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(roc_auc, acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

0.8239789196310936 0.7932960893854749


# 2. 실전 예상문제
제공된 학습용 데이터(car_insur_train.csv)는 차량 소유자의 연령, 성별, 차량 정보 및 보험 여부 등을 포함한 데이터이다. 해당 데이터를 기반으로 보험 가입 여부를 예측하는 분류 모델을 개발하고, 가장 우수한 모델을 평가 데이터(car_insur_test.csv)에 적용하여 생존 예측하시오. 예측 결과는 아래의 [제출 형식]을 준수하여, CSV 파일로 생성하는 코드를 제출하시오.
* 예측결과는 Accuracy 평가지표에 따라 평가함

[제공 데이터]
1. car_insur_train.csv : 학습용 데이터 (3,811건)
2. car_insur_test.csv : 평가용 데이터 (1,270건, 가입 여부 컬럼 미제공)

[데이터 컬럼]
1. id : 고유 식별자
2. Gender : 성별 (Male, Female)
3. Age : 나이
4. Driving_License : 운전면허 보유 여부 (1: 있음, 0: 없음)
5. Region_Code : 지역 코드
6. Previously_Insured : 이전 보험 가입 여부 (1: 있음, 0: 없음)
7. Vehicle_age : 차량 연식 ( < 1 Year, 1-2 Year, > 2 Years)
8. Vehicl_Damage : 과거 차량 손상 이력
9. ... 기타 여러가지 특성
10. Response : 보험 가입 여부 (1: 가입, 0: 미가입)

[제출 형식]
1. 제출 파일명 : result.csv
2. 제출 컬럼명 : pred (0, 1)
3. 예측 결과 개수 : 1,270




In [63]:
# 출력을 원하실 경우 print() 함수 활용
# 예시) print(df.head())

# getcmd(), chdir() 등 작업 폴더 설정 불필요
# 파일 경로 상 내부 드라이버 경로(C: 등) 접근 불가

import pandas as pd

train = pd.read_csv('../data/car_insur_train.csv')
test = pd.read_csv('../data/car_insur_test.csv')

# 답안 제출 참고
# 아래 코드는 예시이며 변수명 등 개인별로 변경하여 활용
# pd.DataFrame변수.to_csv('result.csv', index=False)

# 여기에 코드를 작성하시오.

# print(train.info())
# print(test.info())

# print(train.isnull().sum())
# print(test.isnull().sum())

# print(train['Gender'].value_counts())
# print(train['Region_Code'].value_counts())
# print(train['Vehicle_Age'].value_counts())
# print(train['Vehicle_Damage'].value_counts())

X = train.drop(['id', 'Response'], axis=1)
y = train['Response']

# 여기선 axis=0 주의!!!
X_full = pd.concat([X,test], axis=0)
X_full = X_full.drop('id', axis=1)
# print(X_full.shape)

# 수치형 스케일링 생략

# 범주형 원핫 인코딩 한 번에 
X_full = pd.get_dummies(X_full)
# print(X_full)

X_train = X_full[:train.shape[0]]
test = X_full[train.shape[0]:]

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)

# print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
y_proba = model.predict_proba(X_val)[:,1]

auc = roc_auc_score(y_val, y_proba)
acc = accuracy_score(y_val, y_pred)
print(auc, acc)

y_pred = model.predict(test)
result = pd.DataFrame(y_pred, columns=['pred'])
print(result)
result.to_csv('result_13_2.csv', index = False)




0.8297705023270743 0.8663171690694627
      pred
0        0
1        0
2        0
3        0
4        0
...    ...
1265     0
1266     0
1267     0
1268     0
1269     0

[1270 rows x 1 columns]


# 2.풀이

In [ ]:
import pandas as pd

train = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/car_insur_train.csv')
test = pd.read_csv('/content/drive/MyDrive/BigData/2_DataSet/car_insur_test.csv')

# 데이터 유형 파악
#print(train.info())
#print(test.info())

# 결측치 파악
#print(train.isnull().sum())
#print(test.isnull().sum())

# 범주형 변수 카테고리 파악
#print(train['Gender'].value_counts())
#print(train['Region_Code'].value_counts())
#print(train['Vehicle_Age'].value_counts())
#print(train['Vehicle_Damage'].value_counts())

# X, Y 데이터 셋 분리
X = train.drop(['Response'], axis=1)
y = train['Response']

X_full = pd.concat([X, test], axis=0)
X_full = X_full.drop(['id'], axis=1)
#print(X_full.shape)

# 범주형 변수 인코딩 - One-Hot Encoding
X_full = pd.get_dummies(X_full)
#print(X_full)

# 학습, 검증 데이터 셋 분할
X_train = X_full[:train.shape[0]]
X_test = X_full[train.shape[0]:]
#print(X_train.shape, X_test.shape)

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X_train, y, test_size=0.2, stratify=y)
#print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

# Randomforest 활용 모델 학습
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier()
model.fit(X_train, y_train)

# ROC_AUC, Accuracy 활용 평가
from sklearn.metrics import roc_auc_score, accuracy_score
y_pred = model.predict(X_val)
acc = accuracy_score(y_val, y_pred)
print(acc)

# test 데이터 예측 및 결과 저장 (pred 1개 컬럼)
y_pred = model.predict(X_test)
result = pd.DataFrame(y_pred, columns=['pred'])
result.to_csv('result.csv', index=False)

0.8650065530799476
